<a href="https://colab.research.google.com/github/Jeevan0221/Tj-ai-portfolio/blob/main/day14_web_interface.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
!pip install gradio anthropic


In [10]:
import gradio as gr
import anthropic
import sqlite3
from google.colab import userdata

api_key = userdata.get('ANTHROPIC_KEY')
client = anthropic.Anthropic(api_key=api_key)

print("All tools loaded!")

All tools loaded!


In [11]:
# Create database with employee data
conn = sqlite3.connect('company.db')
cursor = conn.cursor()

cursor.execute('''
    CREATE TABLE IF NOT EXISTS employees (
        id INTEGER PRIMARY KEY,
        name TEXT,
        department TEXT,
        salary INTEGER,
        years_experience INTEGER,
        city TEXT
    )
''')

employees_data = [
    (1, 'John Smith', 'Engineering', 145000, 8, 'New York'),
    (2, 'Sarah Johnson', 'Marketing', 95000, 5, 'Los Angeles'),
    (3, 'Mike Davis', 'Engineering', 165000, 12, 'San Francisco'),
    (4, 'Emily Brown', 'HR', 75000, 3, 'Chicago'),
    (5, 'James Wilson', 'Engineering', 125000, 6, 'New York'),
    (6, 'Lisa Anderson', 'Marketing', 110000, 7, 'Los Angeles'),
    (7, 'Robert Taylor', 'Finance', 135000, 9, 'Chicago'),
    (8, 'Jennifer Martinez', 'HR', 80000, 4, 'San Francisco'),
    (9, 'David Thomas', 'Finance', 155000, 11, 'New York'),
    (10, 'Jessica Garcia', 'Engineering', 140000, 7, 'Chicago'),
]

cursor.executemany('INSERT OR IGNORE INTO employees VALUES (?,?,?,?,?,?)', employees_data)
conn.commit()

print("Database ready with 10 employees!")

Database ready with 10 employees!


In [12]:
def answer_question(question):

    if not question.strip():
        return "Please type a question!"

    try:
        # Create fresh connection inside the function
        conn = sqlite3.connect('company.db')
        cursor = conn.cursor()

        # Step 1 - Ask Claude to write SQL
        sql_prompt = f"""
You are a SQL expert. You have access to a SQLite database with this table:

TABLE: employees
COLUMNS:
- id (INTEGER) - unique employee ID
- name (TEXT) - employee full name
- department (TEXT) - Engineering, Marketing, HR, Finance
- salary (INTEGER) - annual salary in dollars
- years_experience (INTEGER) - years of work experience
- city (TEXT) - New York, Los Angeles, San Francisco, Chicago

USER QUESTION: {question}

Write a single SQLite SQL query to answer this question.
Return ONLY the SQL query. No explanation. No markdown. Just raw SQL.
"""

        sql_response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=256,
            messages=[{"role": "user", "content": sql_prompt}]
        )

        sql_query = sql_response.content[0].text.strip()

        # Step 2 - Run the SQL
        cursor.execute(sql_query)
        results = cursor.fetchall()

        # Step 3 - Ask Claude to explain results
        explain_prompt = f"""
The user asked: {question}
SQL query used: {sql_query}
Database results: {results}

Explain the answer clearly in plain English in 2-3 sentences.
Format it nicely for a non-technical HR manager.
"""

        explain_response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=256,
            messages=[{"role": "user", "content": explain_prompt}]
        )

        final_answer = explain_response.content[0].text

        # Close connection when done
        conn.close()

        return f"""QUESTION: {question}

SQL GENERATED: {sql_query}

ANSWER: {final_answer}"""

    except Exception as e:
        return f"Error: {str(e)}\nTry rephrasing your question."

print("Function updated!")

Function updated!


In [13]:
# Build the web interface
interface = gr.Interface(
    fn=answer_question,
    inputs=gr.Textbox(
        label="Ask a question about employees",
        placeholder="e.g. Who is the highest paid employee?",
        lines=2
    ),
    outputs=gr.Textbox(
        label="Answer",
        lines=10
    ),
    title="AI Employee Database Assistant",
    description="Ask any question about employee data in plain English. No SQL knowledge needed!",
    examples=[
        ["Which department has the highest average salary?"],
        ["How many employees work in New York?"],
        ["Who has the most years of experience?"],
        ["List all employees earning more than $130,000"]
    ]
)

interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://25c566042125dc4910.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
